# 历史分析 Notebook

已清除执行输出和本机凭据。原始数据不随仓库发布。当前可复现入口见根目录 README。可视化 Notebook 的部分查询依赖原数据库中间表，其建表与用户分层规则未完整留档；不将这些查询视为已完成复现。原手填漏斗和汇总图已移除，明确口径的重算结果见 audit_results.json。


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "04_python" else Path.cwd()

# 原始数据
RAW_FILE = PROJECT_DIR / "01_raw_data" / "UserBehavior.csv"

# 之后抽样生成的数据
OUTPUT_FILE = PROJECT_DIR / "02_processed_data" / "UserBehavior_sample.csv"

print("原始数据：", RAW_FILE)
print("文件是否存在：", RAW_FILE.exists())

if RAW_FILE.exists():
    print("文件大小：", round(RAW_FILE.stat().st_size / 1024**3, 2), "GB")

In [ ]:
columns=[
    "user_id",
    "item_id",
    "category_id",
    "behavior_type",
    "timestamp"
]
preview=pd.read_csv(
     RAW_FILE,
    names=columns,
    nrows=10
)
preview

In [ ]:
preview["datetime"]=pd.to_datetime(
    preview["timestamp"],
    unit="s"
)
preview

In [ ]:
# ==============================
# 按 user_id 固定规则抽样
# ==============================

import pandas as pd
import time

columns = [
    "user_id",
    "item_id",
    "category_id",
    "behavior_type",
    "timestamp"
]

# 每次读取100万行，避免一次性加载3.4GB数据
CHUNK_SIZE = 1_000_000

# 淘宝数据约100万用户
# user_id % 20 == 0，大约保留5%的用户，即约5万用户
MOD = 20
REMAINDER = 0

start_time = time.time()

# 如果之前生成过sample文件，先删除，防止重复追加
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

first_chunk = True
total_rows = 0
sample_rows = 0

reader = pd.read_csv(
    RAW_FILE,
    names=columns,
    chunksize=CHUNK_SIZE
)

for i, chunk in enumerate(reader, start=1):

    # 按用户ID筛选
    sample_chunk = chunk[
        chunk["user_id"] % MOD == REMAINDER
    ]

    # 写入CSV
    sample_chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    total_rows += len(chunk)
    sample_rows += len(sample_chunk)

    print(
        f"第 {i} 块完成 | "
        f"已读取 {total_rows:,} 行 | "
        f"已保留 {sample_rows:,} 行"
    )

elapsed = time.time() - start_time

print("\n========== 抽样完成 ==========")
print(f"原始总行数：{total_rows:,}")
print(f"抽样后行数：{sample_rows:,}")
print(f"行抽样比例：{sample_rows / total_rows:.2%}")
print(f"耗时：{elapsed / 60:.2f} 分钟")
print("输出文件：", OUTPUT_FILE)

In [ ]:
sample = pd.read_csv(OUTPUT_FILE)

print("========== 抽样数据基本信息 ==========")
print(f"行为记录数：{len(sample):,}")
print(f"用户数：{sample['user_id'].nunique():,}")
print(f"商品数：{sample['item_id'].nunique():,}")
print(f"类目数：{sample['category_id'].nunique():,}")

print("\n========== 行为类型 ==========")
print(sample["behavior_type"].value_counts())

print("\n========== 缺失值 ==========")
print(sample.isnull().sum())

print("\n========== 重复记录 ==========")
print(sample.duplicated().sum())

print("\n========== 数据预览 ==========")
display(sample.head())

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "04_python" else Path.cwd()

INPUT_FILE = PROJECT_DIR / "02_processed_data" / "UserBehavior_sample.csv"
OUTPUT_FILE = PROJECT_DIR / "02_processed_data" / "UserBehavior_clean.csv"

df = pd.read_csv(INPUT_FILE)

print(f"数据行数：{len(df):,}")
print(f"用户数：{df['user_id'].nunique():,}")

df.head()

In [ ]:
# Unix 时间戳 → 中国标准时间（UTC+8）
df["datetime"] = (
    pd.to_datetime(df["timestamp"], unit="s", utc=True)
      .dt.tz_convert("Asia/Shanghai")
      .dt.tz_localize(None)
)

df[["timestamp", "datetime"]].head()

In [ ]:
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.dayofweek

df[
    ["datetime", "date", "hour", "weekday"]
].head()

In [ ]:
print("最早时间：", df["datetime"].min())
print("最晚时间：", df["datetime"].max())

print("\n每天记录数：")
print(
    df.groupby("date")
      .size()
      .sort_index()
)

In [ ]:
print("行为类型：")
print(df["behavior_type"].value_counts())

valid_behaviors = {"pv", "fav", "cart", "buy"}

invalid_count = (~df["behavior_type"].isin(valid_behaviors)).sum()

print("\n异常行为记录数：", invalid_count)

In [ ]:
# ==============================
# 过滤正式分析时间窗口
# ==============================

START_DATE = pd.Timestamp("2017-11-25 00:00:00")
END_DATE = pd.Timestamp("2017-12-03 23:59:59")

before_rows = len(df)

df_clean = df[
    (df["datetime"] >= START_DATE) &
    (df["datetime"] <= END_DATE)
].copy()

after_rows = len(df_clean)
removed_rows = before_rows - after_rows

print("========== 时间异常清洗 ==========")
print(f"清洗前：{before_rows:,} 行")
print(f"清洗后：{after_rows:,} 行")
print(f"删除异常时间记录：{removed_rows:,} 行")
print(f"删除比例：{removed_rows / before_rows:.4%}")

print("\n清洗后时间范围：")
print("最早：", df_clean["datetime"].min())
print("最晚：", df_clean["datetime"].max())

In [ ]:
# 日期
df_clean["date"] = df_clean["datetime"].dt.date

# 小时：0~23
df_clean["hour"] = df_clean["datetime"].dt.hour

# 星期：0=周一，6=周日
df_clean["weekday"] = df_clean["datetime"].dt.dayofweek

weekday_map = {
    0: "周一",
    1: "周二",
    2: "周三",
    3: "周四",
    4: "周五",
    5: "周六",
    6: "周日"
}

df_clean["weekday_name"] = df_clean["weekday"].map(weekday_map)

# 是否周末
df_clean["is_weekend"] = df_clean["weekday"].isin([5, 6]).astype(int)

df_clean.head()

In [ ]:
print("========== 最终数据质量检查 ==========")

print(f"记录数：{len(df_clean):,}")
print(f"用户数：{df_clean['user_id'].nunique():,}")
print(f"商品数：{df_clean['item_id'].nunique():,}")
print(f"类目数：{df_clean['category_id'].nunique():,}")

print("\n缺失值：")
print(df_clean.isnull().sum())

print("\n重复记录：")
print(df_clean.duplicated().sum())

print("\n行为类型：")
print(df_clean["behavior_type"].value_counts())

print("\n时间范围：")
print(df_clean["datetime"].min())
print(df_clean["datetime"].max())

print("\n每日数据量：")
print(df_clean.groupby("date").size())

In [ ]:
# ==============================
# 保存清洗后的正式分析数据
# ==============================

df_clean.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("保存成功！")
print("文件位置：", OUTPUT_FILE)
print(f"记录数：{len(df_clean):,}")
print(f"用户数：{df_clean['user_id'].nunique():,}")
print(
    "文件大小：",
    round(OUTPUT_FILE.stat().st_size / 1024**2, 2),
    "MB"
)